# Test the Iceberg MCP server connection

Runs inside a **CML Session in the `iceberg-mcp` project**. It connects to the deployed MCP
Application over **streamable HTTP** (endpoint `/mcp/`), authenticating to the Knox gateway with a
Bearer token, then exercises the two tools: `get_schema()` and `execute_query(...)`.

The server talks to the `xom-ice-kudu` CDW Impala VW, so a successful `get_schema()` proves the
whole path: notebook → Knox → MCP app → impyla → Impala.

In [ ]:
# fastmcp is installed in the project's ~/.local by the bootstrap job; install here too if the
# session runtime doesn't see it yet.
try:
    import fastmcp  # noqa
except Exception:
    import sys, subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--user', 'fastmcp>=2.9.2'])
import fastmcp
print('fastmcp', fastmcp.__version__)

In [ ]:
import os

# The deployed app's base URL (no trailing /mcp/). Filled in after deploy; override via env if needed.
APP_URL = os.environ.get(
    'MCP_APP_URL',
    'https://REPLACE-ME.ml-cc27e256-5e1.se-sandb.a465-9q4k.cloudera.site',
)
MCP_ENDPOINT = APP_URL.rstrip('/') + '/mcp/'

# Bearer for the Knox gateway. In a CML session this is provided as CDSW_APIV2_KEY;
# otherwise export CML_API_KEY before launching Jupyter.
TOKEN = os.environ.get('CDSW_APIV2_KEY') or os.environ.get('CML_API_KEY')
assert TOKEN, 'No Bearer token: set CDSW_APIV2_KEY (auto in CML) or CML_API_KEY'
print('endpoint:', MCP_ENDPOINT)

In [ ]:
from fastmcp import Client
from fastmcp.client.transports import StreamableHttpTransport

transport = StreamableHttpTransport(url=MCP_ENDPOINT, headers={'Authorization': f'Bearer {TOKEN}'})

async with Client(transport) as client:
    tools = await client.list_tools()
    print('tools:', [t.name for t in tools])

In [ ]:
# get_schema() -> JSON list of table names in IMPALA_DATABASE (proves Impala connectivity)
async with Client(transport) as client:
    schema = await client.call_tool('get_schema', {})
    print(schema.data if hasattr(schema, 'data') else schema)

In [ ]:
# execute_query(): read-only (select/show/describe/with only). SHOW DATABASES is a safe smoke test.
async with Client(transport) as client:
    res = await client.call_tool('execute_query', {'query': 'SHOW DATABASES'})
    print(res.data if hasattr(res, 'data') else res)